In [6]:
# Upload ZIP
from google.colab import files
uploaded = files.upload()

# Unzip
import zipfile, os
zip_path = list(uploaded.keys())[0]
extract_path = "/content/dataset"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("Dataset extracted!")

# Fix nested folder issue
if "train" not in os.listdir(extract_path):
    extract_path = os.path.join(extract_path, os.listdir(extract_path)[0])

train_path = os.path.join(extract_path, "train")
val_path = os.path.join(extract_path, "val")

# Imports
import tensorflow as tf
from tensorflow.keras import layers, Sequential
import numpy as np

img_size = (180, 180)
batch_size = 32

# Load dataset (with fallback if no val folder)
if os.path.exists(val_path):
    train_data = tf.keras.utils.image_dataset_from_directory(
        train_path,
        image_size=img_size,
        batch_size=batch_size,
        shuffle=True
    )

    val_data = tf.keras.utils.image_dataset_from_directory(
        val_path,
        image_size=img_size,
        batch_size=batch_size
    )
else:
    print("No val folder found → using validation_split")

    train_data = tf.keras.utils.image_dataset_from_directory(
        train_path,
        validation_split=0.2,
        subset="training",
        seed=123,
        image_size=img_size,
        batch_size=batch_size
    )

    val_data = tf.keras.utils.image_dataset_from_directory(
        train_path,
        validation_split=0.2,
        subset="validation",
        seed=123,
        image_size=img_size,
        batch_size=batch_size
    )

class_names = train_data.class_names
print("Classes:", class_names)

# Performance boost
AUTOTUNE = tf.data.AUTOTUNE
train_data = train_data.prefetch(buffer_size=AUTOTUNE)
val_data = val_data.prefetch(buffer_size=AUTOTUNE)

# Improved Model
model = Sequential([
    layers.Rescaling(1./255, input_shape=(180, 180, 3)),

    layers.Conv2D(32, 3, activation='relu', padding='same'),
    layers.MaxPooling2D(),

    layers.Conv2D(64, 3, activation='relu', padding='same'),
    layers.MaxPooling2D(),

    layers.Conv2D(128, 3, activation='relu', padding='same'),
    layers.MaxPooling2D(),

    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dense(len(class_names))
])

model.compile(
    optimizer='adam',
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=['accuracy']
)

# Train
model.fit(train_data, validation_data=val_data, epochs=5)

# Upload test image
uploaded = files.upload()
img_path = list(uploaded.keys())[0]

# Predict
img = tf.keras.utils.load_img(img_path, target_size=img_size)
img_array = tf.keras.utils.img_to_array(img)
img_array = tf.expand_dims(img_array, 0)

prediction = model.predict(img_array)
score = tf.nn.softmax(prediction)

print("Prediction:", class_names[np.argmax(score)])
print("Confidence: {:.2f}%".format(100 * np.max(score)))

Saving dataset.zip to dataset.zip
Dataset extracted!
Found 4 files belonging to 2 classes.
Found 2 files belonging to 2 classes.
Classes: ['cats', 'dogs']


/usr/local/lib/python3.12/dist-packages/keras/src/layers/preprocessing/data_layer.py:95: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/5
1/1 ━━━━━━━━━━━━━━━━━━━━ 3s 3s/step - accuracy: 0.7500 - loss: 0.6572 - val_accuracy: 0.5000 - val_loss: 7.1983
Epoch 2/5
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 450ms/step - accuracy: 0.7500 - loss: 1.4585 - val_accuracy: 0.5000 - val_loss: 5.0438
Epoch 3/5
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 621ms/step - accuracy: 0.7500 - loss: 0.3450 - val_accuracy: 0.5000 - val_loss: 1.9206
Epoch 4/5
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 429ms/step - accuracy: 1.0000 - loss: 0.0885 - val_accuracy: 0.5000 - val_loss: 1.5085
Epoch 5/5
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 642ms/step - accuracy: 1.0000 - loss: 0.1308 - val_accuracy: 0.5000 - val_loss: 2.7587


Saving download (4).jpg to download (4).jpg
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 127ms/step
Prediction: cats
Confidence: 89.43%
